<a href="https://colab.research.google.com/github/fardinAhmed285/tsp-crossover-study/blob/main/tsp_crossover_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import urllib.request

# Download ulysses22.tsp from a reliable GitHub repository
url = "https://raw.githubusercontent.com/mastqe/tsplib/master/ulysses22.tsp"
urllib.request.urlretrieve(url, "ulysses22.tsp")

print("Downloaded ulysses22.tsp successfully!")

Downloaded ulysses22.tsp successfully!


In [2]:
!pip install deap tsplib95

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 37.1 MB/s  0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 2.5.1
    Uninstalling wrapt-2.5.1:
      Successfully uninstalled wrapt-2.5.1
  Attempting uninstall: tabulate
    Found existing installation: tabulate 0.9.0
    Uninstalling tabulate-0.9.0:
      Successfully uninstalled tabulate-0.9.0
  Attempting uninstall: networkx
    Found existing installation: networkx 3.7
    Uninstalling networkx-3.7:
      Successfully uninstalled networkx-3.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [deap]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
momepy 0.11.0 requires networkx>=3.2, but you have networkx 2.8.8 which is incompatible.
spopt 0.7.0 requires networkx>=3.2, but you have networkx 2.8.8 whi

In [8]:
import random
import numpy as np
from deap import base, creator, tools, algorithms
import tsplib95

# 1. Load the ulysses22 problem file
problem = tsplib95.load("ulysses22.tsp")
original_nodes = list(problem.get_nodes())
n_cities = len(original_nodes)

# 2. Set up DEAP framework for Minimization
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("indices", random.sample, range(n_cities), n_cities)
toolbox.register("individual", tools.initIterate, creator.Individual, toolbox.indices)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# 3. Define Fitness Evaluation
def eval_tsp(individual):
    total_distance = 0
    for i in range(len(individual)):
        u = original_nodes[individual[i]]
        v = original_nodes[individual[(i + 1) % len(individual)]]
        total_distance += problem.get_weight(u, v)
    return (total_distance,)

toolbox.register("evaluate", eval_tsp)

# 4. Helper Functions for Edge Heritability Tracking
def get_edges(route):
    edges = set()
    n = len(route)
    for i in range(n):
        u = route[i]
        v = route[(i + 1) % n]
        edges.add(tuple(sorted((u, v))))
    return edges

# List to keep track of heritability scores across generations
heritability_log = []

def custom_mate_with_heritability(ind1, ind2):
    # Make copies of parents before mating (since mate modifies them in place)
    p1_copy = list(ind1)
    p2_copy = list(ind2)

    # Perform standard DEAP Order Crossover (OX)
    tools.cxOrdered(ind1, ind2)

    # Calculate heritability for both children produced
    h1 = calculate_heritability(ind1, p1_copy, p2_copy)
    h2 = calculate_heritability(ind2, p1_copy, p2_copy)

    heritability_log.extend([h1, h2])
    return ind1, ind2

def calculate_heritability(child, p1, p2):
    p1_edges = get_edges(p1)
    p2_edges = get_edges(p2)
    child_edges = get_edges(child)
    parent_pool = p1_edges.union(p2_edges)
    inherited = sum(1 for edge in child_edges if edge in parent_pool)
    return inherited / len(child_edges)

# Register Operators (Using OX)
toolbox.register("mate", custom_mate_with_heritability)
toolbox.register("mutate", tools.mutShuffleIndexes, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

# 5. Run the Evolutionary Loop
def main():
    random.seed(42)
    pop = toolbox.population(n=100)
    hof = tools.HallOfFame(1)

    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("min", np.min)

    print("Running genetic algorithm and tracking edge heritability...")
    pop, logbook = algorithms.eaSimple(pop, toolbox, cxpb=0.9, mutpb=0.1, ngen=50,
                                       stats=stats, halloffame=hof, verbose=True)

    # Print average edge heritability captured during the run
    avg_heritability = np.mean(heritability_log) * 100
    print("\n--- Edge Heritability Results ---")
    print(f"Average Edge Heritability (OX Crossover): {avg_heritability:.2f}%")

    best_route_mapped = [original_nodes[i] for i in hof[0]]
    print(f"Best Distance: {hof[0].fitness.values[0]} (Optimal: 7013)")

if __name__ == "__main__":
    main()

Running genetic algorithm and tracking edge heritability...
gen	nevals	min  
0  	100   	13210
1  	94    	12941
2  	89    	11649
3  	94    	11388
4  	92    	11303
5  	98    	11444
6  	93    	11011
7  	97    	10455
8  	96    	9823 
9  	91    	9471 
10 	92    	9897 
11 	89    	9815 
12 	80    	9897 
13 	90    	9897 
14 	88    	9883 
15 	89    	9750 
16 	90    	9782 
17 	97    	9530 
18 	87    	8927 
19 	94    	9173 
20 	88    	8883 
21 	95    	9014 
22 	88    	9002 
23 	93    	8927 
24 	94    	8993 
25 	90    	8901 
26 	91    	8870 
27 	92    	8701 
28 	91    	8566 
29 	95    	8369 
30 	89    	8636 
31 	95    	8385 
32 	83    	8232 
33 	95    	8571 
34 	88    	8395 
35 	94    	8274 
36 	97    	8330 
37 	83    	8330 
38 	90    	8243 
39 	93    	8326 
40 	90    	8290 
41 	94    	8141 
42 	91    	8097 
43 	88    	8041 
44 	93    	8054 
45 	91    	8054 
46 	90    	7938 
47 	97    	7882 
48 	93    	7882 
49 	89    	7937 
50 	89    	7933 

--- Edge Heritability Results ---
Average Edge Heritabi